<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/08_known_drugs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q anthropic pydantic tenacity biopython
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, time, glob, zipfile, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
os.makedirs(f'{BASE}/data/processed/llm_drugs', exist_ok=True)

hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')
vcap_names = set(vcap.loc[~vcap['pert_iname'].str.startswith('BRD-'), 'pert_iname'].str.lower())
print(len(vcap_names), 'named compounds in VCAP')

def norm(name):
    return str(name).strip().lower().replace(' ', '-')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.9 MB/s eta 0:00:00
Mounted at /content/drive
2966 named compounds in VCAP


In [2]:
!wget -q -O {BASE}/data/raw/drugage.zip https://genomics.senescence.info/drugs/dataset.zip
with zipfile.ZipFile(f'{BASE}/data/raw/drugage.zip') as z:
    z.extractall(f'{BASE}/data/raw/drugage')
da = pd.read_csv(glob.glob(f'{BASE}/data/raw/drugage/*.csv')[0])
print(da.shape)
print(da.columns.tolist())
da.head()

(3423, 15)
['compound_name', 'species', 'strain', 'dosage', 'age_at_initiation', 'treatment_duration', 'avg_lifespan_change_percent', 'avg_lifespan_significance', 'max_lifespan_change_percent', 'max_lifespan_significance', 'gender', 'weight_change_percent', 'weight_change_significance', 'ITP', 'pubmed_id']


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
0,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,81.29,NaN,NaN,NaN,Male,NaN,NaN,No,13369
1,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,50.66,NaN,NaN,NaN,Male,NaN,NaN,No,13369
2,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,71.03,NaN,NaN,NaN,Female,NaN,NaN,No,13369
3,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,123.26,NaN,NaN,NaN,Female,NaN,NaN,No,13369
4,Ethanol,Drosophila mojavensis,A512d,4%,NaN,NaN,20.88,NaN,NaN,NaN,Male,NaN,NaN,No,13369


In [5]:
SYNONYMS = {'rapamycin': 'sirolimus'}   # DrugAge name -> LINCS name
da['name'] = da['compound_name'].map(norm).replace(SYNONYMS)
da['change'] = pd.to_numeric(da['avg_lifespan_change_percent'], errors='coerce')

sig_col = [c for c in da.columns if 'signif' in c.lower()][0]
print('Using column:', sig_col, '| values:', da[sig_col].unique()[:10])
sig = da[da[sig_col].astype(str).str.strip().str.upper() == 'S']
hits = sig[sig['name'].isin(vcap_names)]
drugage = (hits.groupby('name')
           .agg(studies=('name', 'size'),
                species=('species', lambda s: ', '.join(sorted(set(s)))),
                best_change=('change', 'max'))
           .sort_values('studies', ascending=False))
print(len(drugage), 'DrugAge compounds with significant lifespan results are in VCAP')
drugage.to_csv(f'{BASE}/data/processed/drugage_in_vcap.csv')
drugage.head(20)

Using column: avg_lifespan_significance | values: [nan 'NS' 'S']
139 DrugAge compounds with significant lifespan results are in VCAP


,studies,species,best_change
name,,,
resveratrol,57,"Aedes aegypti, Anastrepha ludens, Anopheles st...",70.00
quercetin,33,"Aedes aegypti, Aedes albopictus, Caenorhabditi...",59.85
metformin,31,"Acheta domesticus, Caenorhabditis elegans, Dro...",79.00
sirolimus,31,"Caenorhabditis elegans, Drosophila melanogaste...",38.40
caffeine,28,"Caenorhabditis elegans, Drosophila melanogaste...",86.00
curcumin,23,"Caenorhabditis elegans, Drosophila melanogaster",49.00
aspirin,18,"Acheta domesticus, Caenorhabditis elegans, Dro...",134.00
genistein,13,"Aedes aegypti, Caenorhabditis elegans, Drosoph...",61.50
rifampicin,12,"Caenorhabditis elegans, Drosophila melanogaster",56.10


In [6]:
import anthropic
from pydantic import BaseModel
from typing import Literal
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

class KnownDrug(BaseModel):
    drug: str
    effect: Literal['helpful', 'harmful']
    mechanism: str

class HallmarkDrugs(BaseModel):
    hallmark: str
    drugs: list[KnownDrug]

HELPFUL = {'senescence': 'reduces senescence or the senescence-associated secretory phenotype',
           'nutrient_sensing': 'inhibits mTORC1 signaling',
           'mitochondria': 'increases mitochondrial oxidative phosphorylation or biogenesis',
           'autophagy': 'induces autophagy',
           'inflammation': 'suppresses NF-kB or inflammatory signaling',
           'proteostasis': 'reduces protein-folding stress'}
HARMFUL = {'senescence': 'induces senescence',
           'nutrient_sensing': 'activates mTORC1 signaling',
           'mitochondria': 'inhibits mitochondrial respiration',
           'autophagy': 'blocks autophagy',
           'inflammation': 'triggers inflammatory signaling',
           'proteostasis': 'causes protein-folding stress'}

SYSTEM = """You are an expert pharmacologist in aging biology. For the process given, list well-established drugs or
research compounds with a direct, well-documented effect on that process in human cells: 10 to 20 that are helpful
and 10 to 20 that are harmful, as defined in the request. Prefer compounds commonly used in cell-culture experiments.
Use generic names. Respond with only a JSON object with keys hallmark and drugs (a list of objects with keys drug,
effect ('helpful' or 'harmful'), and mechanism, one sentence)."""

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def propose_drugs(name):
    prompt = f'Process: {name}\nHelpful means it {HELPFUL[name]}.\nHarmful means it {HARMFUL[name]}.'
    msg = client.messages.create(model='claude-sonnet-5', max_tokens=8000, system=SYSTEM,
                                 messages=[{'role': 'user', 'content': prompt}])
    raw = next(b.text for b in msg.content if b.type == 'text')
    raw = raw[raw.find('{'): raw.rfind('}') + 1]
    return HallmarkDrugs.model_validate_json(raw)

for name in hallmarks:
    path = f'{BASE}/data/processed/llm_drugs/{name}.json'
    if not os.path.exists(path):
        open(path, 'w').write(propose_drugs(name).model_dump_json(indent=2))
        print(name, 'done')

senescence done
nutrient_sensing done
mitochondria done
autophagy done
inflammation done
proteostasis done


In [7]:
from Bio import Entrez
Entrez.email = 'your.email@example.com'
TERMS = {'senescence': 'senescence', 'nutrient_sensing': 'mTORC1', 'mitochondria': 'mitochondrial respiration',
         'autophagy': 'autophagy', 'inflammation': 'NF-kappaB', 'proteostasis': 'unfolded protein response'}

rows = []
for name in hallmarks:
    for d in json.load(open(f'{BASE}/data/processed/llm_drugs/{name}.json'))['drugs']:
        n = SYNONYMS.get(norm(d['drug']), norm(d['drug']))
        if n in vcap_names:
            q = f'"{d["drug"]}"[Title/Abstract] AND "{TERMS[name]}"[Title/Abstract]'
            count = int(Entrez.read(Entrez.esearch(db='pubmed', term=q, retmax=0))['Count'])
            rows.append({'hallmark': name, 'drug': n, 'effect': d['effect'], 'papers': count, 'mechanism': d['mechanism']})
            time.sleep(0.4)

cands = pd.DataFrame(rows).sort_values(['hallmark', 'effect', 'papers'], ascending=[True, True, False])
cands.to_csv(f'{BASE}/data/processed/known_drug_candidates.csv', index=False)
pd.set_option('display.max_colwidth', 80)
cands

,hallmark,drug,effect,papers,mechanism
51,autophagy,chloroquine,harmful,3770,"Raises lysosomal pH, blocking autophagosome-lysosome fusion and degradation."
52,autophagy,wortmannin,harmful,401,Inhibits PI3K/Vps34 activity required for autophagosome nucleation.
53,autophagy,vinblastine,harmful,82,Disrupts microtubules required for autophagosome-lysosome fusion and transport.
44,autophagy,sirolimus,helpful,9294,"Inhibits mTORC1, relieving its suppression of ULK1 and autophagy initiation."
48,autophagy,resveratrol,helpful,917,Activates SIRT1 and AMPK signaling to promote autophagosome formation.
...,...,...,...,...,...
8,senescence,apigenin,helpful,50,"Inhibits IL-6 secretion and NF-κB activity, acting as a senomorphic."
5,senescence,ruxolitinib,helpful,31,JAK1/2 inhibitor that blocks STAT3-driven SASP gene expression.
3,senescence,everolimus,helpful,26,mTOR inhibitor analog reducing SASP cytokine production.
10,senescence,simvastatin,helpful,25,"Inhibits the mevalonate pathway, reducing NF-κB-driven SASP expression."


In [8]:
known = {
    'senescence':       {'helpful': ['navitoclax', 'quercetin', 'metformin', 'resveratrol'], 'harmful': []},
    'nutrient_sensing': {'helpful': ['sirolimus'], 'harmful': []},
    'mitochondria':     {'helpful': [], 'harmful': []},
    'autophagy':        {'helpful': [], 'harmful': []},
    'inflammation':     {'helpful': [], 'harmful': []},
    'proteostasis':     {'helpful': [], 'harmful': ['tunicamycin', 'thapsigargin', 'bortezomib',
                                                    'geldanamycin', 'tanespimycin', 'mg-132']},
}

known_drugs = {'by_hallmark': known, 'drugage': drugage.index.tolist()}
json.dump(known_drugs, open(f'{BASE}/data/processed/known_drugs.json', 'w'), indent=2)
for name, k in known.items():
    print(f"{name:17} helpful: {len(k['helpful']):2}  harmful: {len(k['harmful']):2}")

senescence        helpful:  4  harmful:  0
nutrient_sensing  helpful:  1  harmful:  0
mitochondria      helpful:  0  harmful:  0
autophagy         helpful:  0  harmful:  0
inflammation      helpful:  0  harmful:  0
proteostasis      helpful:  0  harmful:  6


In [9]:
pd.set_option('display.max_rows', 200)
short = cands.assign(mechanism=cands['mechanism'].str[:70])
print(short.to_string(index=False))

        hallmark            drug  effect  papers                                                              mechanism
       autophagy     chloroquine harmful    3770 Raises lysosomal pH, blocking autophagosome-lysosome fusion and degrad
       autophagy      wortmannin harmful     401    Inhibits PI3K/Vps34 activity required for autophagosome nucleation.
       autophagy     vinblastine harmful      82 Disrupts microtubules required for autophagosome-lysosome fusion and t
       autophagy       sirolimus helpful    9294 Inhibits mTORC1, relieving its suppression of ULK1 and autophagy initi
       autophagy     resveratrol helpful     917 Activates SIRT1 and AMPK signaling to promote autophagosome formation.
       autophagy       metformin helpful     865 Activates AMPK, which phosphorylates ULK1 and inhibits mTORC1, inducin
       autophagy      everolimus helpful     164                   Rapalog that inhibits mTORC1 to stimulate autophagy.
       autophagy   valproic-acid helpful

In [10]:
known = {
    'senescence': {
        'helpful': ['resveratrol', 'metformin', 'curcumin', 'apigenin', 'ruxolitinib', 'simvastatin'],
        'harmful': ['doxorubicin', 'etoposide', 'palbociclib', 'temozolomide', 'camptothecin', 'mitomycin-c']},
    'nutrient_sensing': {
        'helpful': ['sirolimus', 'everolimus', 'temsirolimus', 'pp242', 'ku-0063794', 'metformin', 'resveratrol', 'curcumin'],
        'harmful': []},
    'mitochondria': {
        'helpful': ['resveratrol', 'quercetin', 'rosiglitazone', 'bezafibrate'],
        'harmful': ['rotenone', 'antimycin-a']},
    'autophagy': {
        'helpful': ['sirolimus', 'everolimus', 'pp242', 'resveratrol', 'metformin', 'carbamazepine'],
        'harmful': []},
    'inflammation': {
        'helpful': ['curcumin', 'dexamethasone', 'resveratrol', 'quercetin', 'sirolimus', 'bay-11-7082',
                    'metformin', 'aspirin', 'parthenolide', 'sulfasalazine', 'simvastatin', 'ruxolitinib'],
        'harmful': ['doxorubicin', 'etoposide', 'thapsigargin', 'tunicamycin']},
    'proteostasis': {
        'helpful': ['sirolimus', 'metformin', 'resveratrol'],
        'harmful': ['tunicamycin', 'thapsigargin', 'bortezomib', 'geldanamycin', 'tanespimycin',
                    'mg-132', 'brefeldin-a', 'chloroquine', 'puromycin']},
}

known_drugs = {'by_hallmark': known, 'drugage': drugage.index.tolist()}
json.dump(known_drugs, open(f'{BASE}/data/processed/known_drugs.json', 'w'), indent=2)
for name, k in known.items():
    print(f"{name:17} helpful: {len(k['helpful']):2}  harmful: {len(k['harmful']):2}")

senescence        helpful:  6  harmful:  6
nutrient_sensing  helpful:  8  harmful:  0
mitochondria      helpful:  4  harmful:  2
autophagy         helpful:  6  harmful:  0
inflammation      helpful: 12  harmful:  4
proteostasis      helpful:  3  harmful:  9


In [11]:
import json
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))

all_drugs = {d for k in kd['by_hallmark'].values() for d in k['helpful'] + k['harmful']}
print('N =', len(all_drugs))
print('M =', len(kd['drugage']))

N = 37
M = 139
